# 📦 Step 1 — Download Qwen3.6 Model & Create Kaggle Dataset

Run this notebook **once**. It downloads the Qwen3.6-35B-A3B GGUF model files from HuggingFace directly onto Kaggle's servers, then packages them into a **permanent private Kaggle Dataset**.

> 🔧 Username is auto-detected — no manual edit required.

### What gets downloaded
| File | Size | Purpose |
|---|---|---|
| `Qwen3.6-35B-A3B-UD-Q4_K_XL.gguf` | ~22.4 GB | Main language model (4-bit quantized) |
| `mmproj-F16.gguf` | ~1 GB | Vision projector for multimodal (image) support |

### Prerequisites
- **Internet**: ON
- **Accelerator**: None (CPU is fine)
- Kaggle account with enough free storage (100 GB free)

## ⚙️ Cell 1 — Install dependencies

In [ ]:
%%capture
!pip install -q huggingface_hub hf_transfer

## ⚙️ Cell 2 — Enable fast transfer & define paths

`HF_HUB_ENABLE_HF_TRANSFER=1` activates a Rust transfer backend (3-5x faster).
We download to `/tmp/model/` instead of `/kaggle/working/` because Kaggle enforces a 20 GB quota on `/kaggle/working` — the model alone is 22.4 GB.

In [ ]:
import os
import json
from pathlib import Path

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# ── Configuration ──────────────────────────────────────────────────────────────
REPO_ID      = "unsloth/Qwen3.6-35B-A3B-GGUF"   # HuggingFace model repo
QUANT        = "UD-Q4_K_XL"                       # Quantization variant
DOWNLOAD_DIR = Path("/tmp/model")                  # Where files land
DATASET_SLUG = "qwen36-35b-a3b-gguf"              # Your Kaggle dataset slug
# ───────────────────────────────────────────────────────────────────────────────

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Download directory: {DOWNLOAD_DIR}")
print(f"🤗 Source repo      : {REPO_ID}")
print(f"🗜️  Quantization     : {QUANT}")

## 📥 Cell 3 — Download model files from HuggingFace

Uses `snapshot_download` with glob filters to grab only the two files needed.

In [ ]:
from huggingface_hub import snapshot_download

print("🚀 Starting download from HuggingFace...")
print("   This will take 15–30 minutes. You can monitor progress below.\n")

snapshot_download(
    repo_id   = REPO_ID,
    repo_type = "model",
    local_dir = str(DOWNLOAD_DIR),
    allow_patterns = [
        f"*{QUANT}*",   # Main model (~22.4 GB)
        "*mmproj-F16*"  # Vision projector (~1 GB)
    ],
)
print("\n✅ Download complete.")

## 🔍 Cell 4 — Verify downloaded files

In [ ]:
print("📋 Downloaded files:\n")

total_size = 0
found_model  = False
found_mmproj = False

for f in sorted(DOWNLOAD_DIR.rglob("*.gguf")):
    size_gb = f.stat().st_size / (1024 ** 3)
    total_size += size_gb
    tag = ""
    if QUANT in f.name:
        found_model = True
        tag = "← main model"
    if "mmproj" in f.name:
        found_mmproj = True
        tag = "← vision projector"
    print(f"  {f.name:<55} {size_gb:6.2f} GB  {tag}")

print(f"\n  Total: {total_size:.2f} GB")
print()

assert found_model,  "❌ Main model file not found! Check the download."
assert found_mmproj, "❌ mmproj file not found! Multimodal will not work."
print("✅ Both files verified — ready to create the Kaggle Dataset.")

## 📦 Cell 5 — Create the Kaggle Dataset

Kaggle requires a `dataset-metadata.json`. We generate it using your auto-detected username, then publish the dataset as **private** to your account.

In [ ]:
import subprocess

# ── Your Kaggle username (auto-detected) ───────────────────────────────────────
KAGGLE_USERNAME = ""   # leave "" to auto-detect; or fill in e.g. "johndoe"

# ── Auto-detect Kaggle username (no manual edit needed) ────────────────────────
def _detect_kaggle_username():
    u = os.environ.get("KAGGLE_USERNAME", "").strip()
    if u:
        return u
    u = os.environ.get("KAGGLE_USER", "").strip()
    if u:
        return u
    # Kaggle API token file contains the username — most reliable in headless runs.
    for token in (Path.home() / ".kaggle" / "kaggle.json", Path("/root/.kaggle/kaggle.json")):
        try:
            if token.exists():
                data = json.loads(token.read_text())
                if data.get("username"):
                    return data["username"]
        except Exception:
            pass
    return ""

if not KAGGLE_USERNAME:
    KAGGLE_USERNAME = _detect_kaggle_username()
if not KAGGLE_USERNAME:
    raise ValueError(
        "Could not auto-detect Kaggle username. Set it manually at the top: "
        "KAGGLE_USERNAME = 'your_username'  (find it at https://www.kaggle.com/settings)"
    )

kaggle_username = KAGGLE_USERNAME

print(f"👤 Kaggle username: {kaggle_username}")
print(f"📦 Dataset will be created as: {kaggle_username}/{DATASET_SLUG}")
print()

metadata = {
    "title"   : "Qwen3.6-35B-A3B GGUF",
    "id"      : f"{kaggle_username}/{DATASET_SLUG}",
    "licenses": [{"name": "apache-2.0"}],
    "isPrivate": True
}
metadata_path = DOWNLOAD_DIR / "dataset-metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
print("📝 dataset-metadata.json created.")

print("⬆️  Uploading to Kaggle (this may take several minutes)...\n")
result = subprocess.run(
    ["kaggle", "datasets", "create", "-p", str(DOWNLOAD_DIR), "--dir-mode", "skip"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("⚠️ stderr:", result.stderr)
else:
    print(f"\n✅ Dataset created successfully!")
    print(f"🔗 https://www.kaggle.com/datasets/{kaggle_username}/{DATASET_SLUG}")

## ✅ Done!

Your model files are now stored permanently in your Kaggle account.

**Next:** open `2_run_server.ipynb`, add this dataset as an input, and click **Run All**.